# SoccerNet Download To Google Drive;

Goal: download SoccerNet tracking subset directly to Google Drive persistent storage;

Notes: this notebook is designed to avoid temporary-only `/content` storage;

Run order: Cell 1 -> Cell 2 -> Cell 3 -> Cell 4;

In [6]:
from pathlib import Path
import os
import time

from google.colab import drive  # type: ignore

mount_path = '/content/drive'


def _mount_drive(force_remount: bool) -> tuple[bool, Exception | None]:
    try:
        drive.mount(mount_path, force_remount=force_remount)
        return True, None
    except Exception as error:
        return False, error


if os.path.ismount(mount_path):
    print('Google Drive is already mounted;')
else:
    success, first_error = _mount_drive(force_remount=False)
    if not success:
        print(f'Initial mount attempt failed: {first_error}')
        success, second_error = _mount_drive(force_remount=True)
        if not success:
            raise RuntimeError(
                'Google Drive mount failed after retry. '
                'Open the notebook in browser, ensure third-party cookies are allowed for Google/Colab, '
                'then run this cell again;'
            ) from second_error

if not os.path.ismount(mount_path):
    raise RuntimeError('Google Drive is not mounted as a real filesystem;')

mydrive_root = Path('/content/drive/MyDrive')
if not mydrive_root.exists():
    raise RuntimeError('MyDrive path is not visible after mount;')

probe_file = mydrive_root / 'reid_cache_mount_probe.txt'
probe_file.parent.mkdir(parents=True, exist_ok=True)
probe_file.write_text(f'probe_ts={time.time()}\n', encoding='utf-8')

print('Drive mount validation: OK;')
print(f'Probe file: {probe_file};')

Google Drive is already mounted;
Drive mount validation: OK;
Probe file: /content/drive/MyDrive/reid_cache_mount_probe.txt;


In [8]:
import os
from pathlib import Path

# Required only if your SoccerNet access needs a password.
SOCCERNET_PASSWORD = os.environ.get('SOCCERNET_PASSWORD', '').strip() or None

# Downloads go directly to the connected account Google Drive (MyDrive).
# Optional override example:
#   %env DRIVE_TARGET_PATH=/content/drive/MyDrive/reid_cache/SoccerNet
mydrive_root = Path('/content/drive/MyDrive')
default_target = mydrive_root / 'SoccerNet'
raw_target = os.environ.get('DRIVE_TARGET_PATH', '').strip()

candidate_target = Path(raw_target).expanduser() if raw_target else default_target

try:
    candidate_target.relative_to(mydrive_root)
    target_root = candidate_target
except ValueError:
    print(
        'Warning: DRIVE_TARGET_PATH is outside /content/drive/MyDrive; '
        'falling back to /content/drive/MyDrive/SoccerNet;'
    )
    target_root = default_target

os.environ['DRIVE_TARGET_PATH'] = str(target_root)
target_root.mkdir(parents=True, exist_ok=True)

print(f'Target root: {target_root};')
print(f'SoccerNet password set: {SOCCERNET_PASSWORD is not None};')

Target root: /content/drive/MyDrive/SoccerNet;
SoccerNet password set: False;


In [9]:
# pyright: reportMissingImports=false
import importlib.util
import inspect
import subprocess
import sys

if importlib.util.find_spec('SoccerNet') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'SoccerNet'], check=True)

from SoccerNet.Downloader import SoccerNetDownloader

def _invoke_with_supported_kwargs(method, kwargs):
    signature = inspect.signature(method)
    filtered_kwargs = {
        key: value
        for key, value in kwargs.items()
        if key in signature.parameters and value is not None
    }
    return method(**filtered_kwargs)

downloader = SoccerNetDownloader(LocalDirectory=str(target_root))

method_candidates = [
    (
        'downloadDataTask',
        {
            'task': 'tracking',
            'split': ['train', 'test'],
            'password': SOCCERNET_PASSWORD,
        },
    ),
    (
        'downloadGames',
        {
            'task': ['tracking'],
            'split': ['train', 'test'],
            'password': SOCCERNET_PASSWORD,
        },
    ),
    (
        'download',
        {
            'task': 'tracking',
            'split': ['train', 'test'],
            'password': SOCCERNET_PASSWORD,
        },
    ),
]

errors = []
for method_name, kwargs in method_candidates:
    method = getattr(downloader, method_name, None)
    if method is None:
        continue

    try:
        _invoke_with_supported_kwargs(method, kwargs)
        print(f'Download method used: {method_name};')
        break
    except Exception as error:
        errors.append(f'{method_name}: {error}')
else:
    details = '; '.join(errors) if errors else 'No supported SoccerNet download method was found'
    raise RuntimeError(f'Unable to download SoccerNet tracking subset; {details};')

print('SoccerNet download finished;')
print(f'Destination: {target_root};')

KeyboardInterrupt: 

In [10]:
import subprocess
from pathlib import Path

resolved_target = target_root.resolve()

try:
    resolved_target.relative_to(mydrive_root)
    stored_on_drive = True
except ValueError:
    stored_on_drive = False

if not stored_on_drive:
    raise RuntimeError(
        f'Dataset path is not on Google Drive: {resolved_target}; '
        'expected inside /content/drive/MyDrive;'
    )

zip_files = sorted(target_root.rglob('*.zip'))
img_dirs = sorted(target_root.rglob('img1'))
json_files = sorted(target_root.rglob('*.json'))
csv_files = sorted(target_root.rglob('*.csv'))
txt_files = sorted(target_root.rglob('*.txt'))

content_soccernet_dirs = []
content_root = Path('/content')
if content_root.exists():
    content_soccernet_dirs = [
        path
        for path in content_root.iterdir()
        if path.is_dir()
        and 'soccernet' in path.name.lower()
        and not str(path.resolve()).startswith('/content/drive/')
    ]

print('Google Drive storage check: OK;')
print(f'Dataset root: {resolved_target};')
print(f'Zip files: {len(zip_files)};')
print(f'img1 directories: {len(img_dirs)};')
print(f'JSON files: {len(json_files)};')
print(f'CSV files: {len(csv_files)};')
print(f'TXT files: {len(txt_files)};')

if content_soccernet_dirs:
    print('Warning: found SoccerNet-like folders outside Google Drive under /content;')
    for path in content_soccernet_dirs:
        print(f'- {path};')
else:
    print('No SoccerNet-like folders found in ephemeral /content root;')

subprocess.run(
    ['bash', '-lc', f"du -sh '{resolved_target.as_posix()}' 2>/dev/null || true"],
    check=False,
)

print('Done; use this path in future sessions:')
print(f"%env SOCCERNET_ROOT_DIR={resolved_target.as_posix()}")

Google Drive storage check: OK;
Dataset root: /content/drive/MyDrive/SoccerNet;
Zip files: 0;
img1 directories: 0;
JSON files: 0;
CSV files: 0;
TXT files: 0;
No SoccerNet-like folders found in ephemeral /content root;
Done; use this path in future sessions:
%env SOCCERNET_ROOT_DIR=/content/drive/MyDrive/SoccerNet


In [ ]:
import os
import shutil
import time
from pathlib import Path

# Optional override for destination folder on your Google Drive:
#   %env DRIVE_COPY_TARGET_PATH=/content/drive/MyDrive/reid_cache/SoccerNet_backup
copy_target_raw = os.environ.get('DRIVE_COPY_TARGET_PATH', '').strip()
default_copy_target = mydrive_root / 'SoccerNet_backup'
copy_target = Path(copy_target_raw).expanduser() if copy_target_raw else default_copy_target

source_root = target_root.resolve()

if not source_root.exists() or not any(source_root.iterdir()):
    raise RuntimeError(
        f'Source dataset folder is empty or missing: {source_root}; '
        'run the download cell first;'
    )

try:
    copy_target.relative_to(mydrive_root)
except ValueError:
    raise ValueError('DRIVE_COPY_TARGET_PATH must be inside /content/drive/MyDrive;')

if copy_target.resolve() == source_root:
    raise ValueError('Copy destination must be different from source path;')

copy_target.mkdir(parents=True, exist_ok=True)

print(f'Copy source: {source_root};')
print(f'Copy destination: {copy_target};')

start_time = time.time()
copied_entries = 0

for entry in source_root.iterdir():
    destination = copy_target / entry.name
    if entry.is_dir():
        shutil.copytree(entry, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(entry, destination)
    copied_entries += 1

elapsed = time.time() - start_time
print(f'Copy completed in {elapsed:.1f}s; top-level entries processed: {copied_entries};')
print(f'Backup location: {copy_target};')